# 🛰️ SAR Change Detection Pipeline (NASA / ASF → PyTorch → ONNX)

Pipeline complet pentru Google Colab (GPU: **RTX PRO 6000 Blackwell**):

1. **Date SAR de la NASA** — produse **OPERA L2 RTC-S1** (Sentinel-1, backscatter corectat radiometric + teren, 30 m, GeoTIFF gata de analiză), servite de **ASF DAAC** (Alaska Satellite Facility – centrul NASA pentru date SAR).
2. **Preprocesare** — aliniere pe aceeași grilă, filtru Lee anti-speckle, conversie în dB, normalizare.
3. **Etichete** — pseudo-etichete din metoda clasică *log-ratio + Otsu* + **schimbări sintetice** injectate (self-supervised).
4. **Model** — **Siamese U-Net** (encoder partajat, fuziune pe diferența feature-urilor), antrenat în bf16.
5. **Inferență pe serie temporală** — hartă de schimbare pentru fiecare pereche de date consecutive + hărți agregate (frecvență, prima dată a schimbării).
6. **Export pentru backend-ul Rust** — `model.onnx` (rulat cu crate-ul `ort`), `model_meta.json`, GeoTIFF-uri, **GeoJSON (EPSG:4326)** și `timeline.json` pentru frontend-ul React.

### Ce îți trebuie înainte
* Cont gratuit **NASA Earthdata**: https://urs.earthdata.nasa.gov/users/new
* În profilul Earthdata → *Applications → Authorized Apps* → aprobă **"Alaska Satellite Facility Data Access"** (altfel download-ul dă 401).
* În Colab: 🔑 *Secrets* (bara din stânga) → adaugă `EARTHDATA_USERNAME` și `EARTHDATA_PASSWORD` (sau un `EARTHDATA_TOKEN`). Dacă lipsesc, notebook-ul îți cere parola interactiv.

## 0. Instalare dependențe și verificare GPU

In [ ]:
!pip -q install asf_search rasterio shapely pyproj scikit-image onnx onnxscript onnxruntime-gpu tqdm

In [ ]:
import os, re, json, math, glob, random, time, datetime as dt
from pathlib import Path

import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("torch", torch.__version__, "| CUDA", torch.version.cuda)
assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU (RTX PRO 6000)"
print("GPU:", torch.cuda.get_device_name(0),
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB",
      "| capability", torch.cuda.get_device_capability(0))
# Blackwell (sm_120) necesită PyTorch construit cu CUDA >= 12.8
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
DEVICE = "cuda"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. Configurare

* `AOI_WKT` — zona de interes (poligon WKT, lon/lat). Exemplul implicit: inundațiile din sept. 2024 de la **Slobozia Conachi / Galați**.
* `START` / `END` — intervalul de timp. Sentinel-1 revizitează o zonă la ~12 zile (6 zile din 2025 cu S1C).
* Totul se salvează în Google Drive ca să nu pierzi datele la deconectare.

In [ ]:
AOI_WKT = "POLYGON((27.70 45.55, 28.05 45.55, 28.05 45.80, 27.70 45.80, 27.70 45.55))"
START   = "2024-07-01"
END     = "2024-11-30"

POLS          = ["VV", "VH"]   # polarizări folosite (canale per dată)
MAX_SCENES    = 30             # câte achiziții maxim descărcăm (pe același burst)
CROP_TO_AOI   = True           # decupează burst-ul la AOI (mai rapid); False = tot burst-ul

# normalizare dB -> [0,1]; ACESTEAȘI valori trebuie folosite în backend-ul Rust
DB_RANGE = {"VV": (-25.0, 0.0), "VH": (-32.0, -5.0)}

# antrenare
TILE        = 256
BATCH       = 32
EPOCHS      = 40
STEPS_EPOCH = 200
LR          = 3e-4
MAX_PAIR_GAP = 3               # perechi (t_i, t_j) cu j-i <= 3 pentru antrenare
MIN_CHANGE_DB = 3.0            # prag minim |Δ dB| pentru pseudo-etichete

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/sar_change_detection")
else:
    ROOT = Path("/content/sar_change_detection")
RAW_DIR, OUT_DIR, CKPT_DIR = ROOT/"raw", ROOT/"outputs", ROOT/"checkpoints"
for d in (RAW_DIR, OUT_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)
print("Lucrez în", ROOT)

## 2. Autentificare NASA Earthdata

Credențialele se citesc din *Colab Secrets* — nu le scrie niciodată direct în notebook.

In [ ]:
import asf_search as asf
from getpass import getpass

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

session = asf.ASFSession()
token = _secret("EARTHDATA_TOKEN")
if token:
    session.auth_with_token(token)
else:
    user = _secret("EARTHDATA_USERNAME") or input("Earthdata username: ")
    pwd  = _secret("EARTHDATA_PASSWORD") or getpass("Earthdata password: ")
    session.auth_with_creds(user, pwd)
print("✅ Autentificat la NASA Earthdata / ASF")

## 3. Căutare date SAR (OPERA RTC-S1 la ASF DAAC)

**De ce OPERA RTC-S1 și nu Sentinel-1 GRD/SLC brut?** RTC-S1 este deja calibrat, corectat de teren și georeferențiat pe o grilă UTM fixă de 30 m per *burst* → imaginile aceluiași burst sunt direct comparabile în timp, fără SNAP/ISCE.

Fiecare produs = un *burst* Sentinel-1 la o dată. Alegem burst-ul care acoperă AOI cel mai des (aceeași orbită și geometrie de vizare = comparație corectă).

> Alternative NASA: `earthaccess.search_data(short_name="OPERA_L2_RTC-S1_V1", ...)` (aceleași fișiere), **NISAR** L2 GCOV (banda L, din 2025/2026) prin `asf.search(dataset="NISAR")`, sau produsul gata făcut **OPERA DIST-S1** (alerte de perturbare) pentru validare.

In [ ]:
results = asf.search(
    dataset="OPERA-S1",
    processingLevel="RTC",
    intersectsWith=AOI_WKT,
    start=START, end=END,
    maxResults=2000,
)
print(f"{len(results)} produse RTC-S1 găsite")

def burst_id(p):
    b = p.properties.get("operaBurstID")
    if b: return b
    m = re.search(r"_(T\d{3}-\d{6}-IW\d)_", p.properties["fileID"])
    return m.group(1) if m else None

from collections import defaultdict
by_burst = defaultdict(list)
for p in results:
    by_burst[burst_id(p)].append(p)

for b, ps in sorted(by_burst.items(), key=lambda kv: -len(kv[1]))[:10]:
    print(f"  {b}: {len(ps)} achiziții")

BURST = max(by_burst, key=lambda b: len(by_burst[b]))
products = sorted(by_burst[BURST], key=lambda p: p.properties["startTime"])

# păstrăm o singură achiziție pe zi (pot exista reprocesări duplicate)
uniq = {}
for p in products:
    uniq[p.properties["startTime"][:10]] = p
products = list(uniq.values())
if len(products) > MAX_SCENES:
    idx = np.linspace(0, len(products) - 1, MAX_SCENES).round().astype(int)
    products = [products[i] for i in idx]
print(f"\nBurst ales: {BURST} → {len(products)} date:", [p.properties['startTime'][:10] for p in products])
assert len(products) >= 2, "Prea puține achiziții: mărește intervalul sau AOI"

## 4. Descărcare GeoTIFF-uri (VV, VH)

In [ ]:
def pol_urls(p):
    urls = [p.properties.get("url")] + list(p.properties.get("additionalUrls") or [])
    out = {}
    for u in urls:
        if not u or not u.endswith(".tif"): continue
        for pol in POLS:
            if u.endswith(f"_{pol}.tif"): out[pol] = u
    return out

scene_files = []          # [(date, {pol: path})]
to_download = []
for p in products:
    urls = pol_urls(p)
    if not all(pol in urls for pol in POLS):
        print("⚠️  lipsește o polarizare, sar peste", p.properties["fileID"]); continue
    files = {}
    for pol, u in urls.items():
        f = RAW_DIR / Path(u).name
        files[pol] = f
        if not f.exists(): to_download.append(u)
    scene_files.append((p.properties["startTime"][:10], files))

print(f"De descărcat: {len(to_download)} fișiere")
if to_download:
    asf.download_urls(urls=to_download, path=str(RAW_DIR), session=session, processes=8)
print("✅ Download complet")

## 5. Preprocesare: aliniere, decupare AOI, filtru Lee, dB, normalizare

Rezultat: `STACK` cu forma `[T, 2, H, W]` (float32, valori în [0,1]) și `VALID` `[T, H, W]`.

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds, transform_geom
from rasterio.windows import from_bounds
from shapely import wkt as shp_wkt
from scipy.ndimage import uniform_filter

aoi_geom = shp_wkt.loads(AOI_WKT)

# grila de referință = prima scenă (decupată la AOI)
with rasterio.open(scene_files[0][1][POLS[0]]) as src:
    REF_CRS = src.crs
    if CROP_TO_AOI:
        b = transform_bounds("EPSG:4326", src.crs, *aoi_geom.bounds)
        win = from_bounds(*b, transform=src.transform).round_offsets().round_lengths()
        win = win.intersection(rasterio.windows.Window(0, 0, src.width, src.height))
    else:
        win = rasterio.windows.Window(0, 0, src.width, src.height)
    REF_TRANSFORM = src.window_transform(win)
    H, W = int(win.height), int(win.width)
print("Grilă:", REF_CRS, f"{H}x{W} px @ {REF_TRANSFORM.a:.0f} m")

def lee_filter(img, size=5):
    m  = uniform_filter(img, size)
    m2 = uniform_filter(img * img, size)
    var = np.maximum(m2 - m * m, 0)
    noise = np.mean(var)
    k = var / (var + noise + 1e-12)
    return m + k * (img - m)

def load_on_ref(path):
    dst = np.full((H, W), np.nan, np.float32)
    with rasterio.open(path) as src:
        reproject(rasterio.band(src, 1), dst,
                  src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                  dst_transform=REF_TRANSFORM, dst_crs=REF_CRS, dst_nodata=np.nan,
                  resampling=Resampling.bilinear)
    return dst

def to_norm(lin, pol):
    lo, hi = DB_RANGE[pol]
    db = 10 * np.log10(np.clip(lin, 1e-5, None))
    return np.clip((db - lo) / (hi - lo), 0, 1).astype(np.float32)

DATES, STACK, VALID, DB = [], [], [], []
for date, files in tqdm(scene_files, desc="preprocesare"):
    chans, dbs, valid = [], [], np.ones((H, W), bool)
    for pol in POLS:
        lin = load_on_ref(files[pol])
        v = np.isfinite(lin) & (lin > 0)
        valid &= v
        lin = np.where(v, lin, np.nanmedian(lin[v]) if v.any() else 1e-3).astype(np.float32)
        lin = lee_filter(lin)
        chans.append(to_norm(lin, pol))
        dbs.append(10 * np.log10(np.clip(lin, 1e-5, None)))
    if valid.mean() < 0.5:
        print("⚠️  scenă cu acoperire slabă, sar peste", date); continue
    DATES.append(date); STACK.append(np.stack(chans)); VALID.append(valid); DB.append(np.stack(dbs))

STACK = np.stack(STACK).astype(np.float32)   # [T,2,H,W]
VALID = np.stack(VALID)                       # [T,H,W]
DB    = np.stack(DB).astype(np.float32)       # [T,2,H,W] (pentru pseudo-etichete)
T = len(DATES)
print("STACK", STACK.shape, "| date:", DATES)
np.savez_compressed(OUT_DIR / "stack.npz", stack=STACK, valid=VALID, dates=np.array(DATES))

In [ ]:
# vizualizare rapidă: prima, mijlocul și ultima dată (VV) + RGB multitemporal
def show(ax, img, title, **kw):
    ax.imshow(img, **kw); ax.set_title(title, fontsize=9); ax.axis("off")

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, i in zip(ax[:3], [0, T // 2, T - 1]):
    show(a, STACK[i, 0], f"VV {DATES[i]}", cmap="gray")
rgb = np.stack([STACK[0, 0], STACK[T // 2, 0], STACK[-1, 0]], -1)
show(ax[3], rgb, f"RGB VV: R={DATES[0]} G={DATES[T//2]} B={DATES[-1]}")
plt.tight_layout(); plt.show()

## 6. Pseudo-etichete (metoda clasică log-ratio + Otsu)

Pentru fiecare pereche `(t_i, t_j)`: `Δ = max(|ΔdB_VV|, |ΔdB_VH|)` → prag Otsu (minim `MIN_CHANGE_DB`) → curățare morfologică.
Pixelii fără date = `255` (ignorați în loss).

> Dacă ai etichete reale (ex. hărți de inundații Copernicus EMS, OPERA DIST-S1, adnotări manuale), înlocuiește `LABELS[(i,j)]` cu ele — restul pipeline-ului rămâne identic.

In [ ]:
from skimage.filters import threshold_otsu
from scipy import ndimage as ndi

def pseudo_label(i, j, min_db=MIN_CHANGE_DB, min_px=20):
    d = np.abs(DB[j] - DB[i]).max(0)
    v = VALID[i] & VALID[j]
    thr = max(threshold_otsu(d[v][::7]), min_db) if v.any() else min_db
    m = d > thr
    m = ndi.binary_opening(m, iterations=1)
    m = ndi.binary_closing(m, iterations=1)
    lab, n = ndi.label(m)
    if n:
        sizes = ndi.sum(m, lab, range(1, n + 1))
        m = np.isin(lab, np.where(sizes >= min_px)[0] + 1)
    out = m.astype(np.uint8)
    out[~v] = 255
    return out, thr

PAIRS = [(i, j) for i in range(T) for j in range(i + 1, min(T, i + MAX_PAIR_GAP + 1))]
LABELS = {}
for i, j in tqdm(PAIRS, desc="pseudo-etichete"):
    LABELS[(i, j)], thr = pseudo_label(i, j)
rates = [np.mean(LABELS[p] == 1) for p in PAIRS]
print(f"{len(PAIRS)} perechi | % pixeli schimbați: medie {100*np.mean(rates):.2f}%, max {100*np.max(rates):.2f}%")

k = int(np.argmax(rates)); i, j = PAIRS[k]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
show(ax[0], STACK[i, 0], f"VV {DATES[i]}", cmap="gray")
show(ax[1], STACK[j, 0], f"VV {DATES[j]}", cmap="gray")
show(ax[2], np.where(LABELS[(i, j)] == 255, np.nan, LABELS[(i, j)]), "pseudo-etichetă", cmap="magma")
plt.tight_layout(); plt.show()

## 7. Dataset cu augmentări + schimbări sintetice

* **Split spațial**: ultimele 20% coloane = validare (fără scurgeri de informație între train/val).
* **Augmentări**: flip/rotație, swap t1↔t2 (schimbarea e simetrică), zgomot gaussian, jitter de luminozitate per dată.
* **Schimbări sintetice** (p=0.3): lipim în t2 o pată cu formă aleatoare luată din altă locație/dată → etichetă 1 unde diferența e reală.
* **Perechi fără schimbare** (p=0.1): t1 vs t1 + zgomot → învață invarianța la speckle.

In [ ]:
VAL_COL = int(W * 0.8)

def random_blob(h, w, rng):
    yy, xx = np.mgrid[:h, :w]
    m = np.zeros((h, w), bool)
    for _ in range(rng.integers(1, 4)):
        cy, cx = rng.integers(0, h), rng.integers(0, w)
        ry, rx = rng.integers(8, h // 4), rng.integers(8, w // 4)
        th = rng.uniform(0, np.pi)
        y, x = yy - cy, xx - cx
        u = (x * np.cos(th) + y * np.sin(th)) / rx
        v = (-x * np.sin(th) + y * np.cos(th)) / ry
        m |= (u * u + v * v) < 1
    return m

class ChangeDataset(torch.utils.data.Dataset):
    def __init__(self, split, n_samples, tile=TILE):
        self.split, self.n, self.tile = split, n_samples, tile
        self.c0, self.c1 = (0, VAL_COL) if split == "train" else (VAL_COL, W)
        if split == "val":   # grilă fixă pentru validare reproductibilă
            ys = range(0, max(1, H - tile + 1), tile)
            xs = range(self.c0, max(self.c0 + 1, self.c1 - tile + 1), tile)
            self.items = [(p, y, x) for p in PAIRS for y in ys for x in xs]
            self.n = len(self.items)

    def __len__(self): return self.n

    def _crop(self, i, j, y, x):
        t = self.tile
        a, b = STACK[i, :, y:y+t, x:x+t], STACK[j, :, y:y+t, x:x+t]
        l = LABELS[(i, j)][y:y+t, x:x+t]
        return a.copy(), b.copy(), l.copy()

    def __getitem__(self, idx):
        t = self.tile
        if self.split == "val":
            (i, j), y, x = self.items[idx]
            a, b, l = self._crop(i, j, y, x)
            return self._pad(a, b, l)

        rng = np.random.default_rng()
        i, j = PAIRS[rng.integers(len(PAIRS))]
        y = rng.integers(0, max(1, H - t + 1))
        x = rng.integers(self.c0, max(self.c0 + 1, self.c1 - t + 1))
        a, b, l = self._crop(i, j, y, x)
        a, b, l = self._pad(a, b, l)

        r = rng.random()
        if r < 0.1:                                   # pereche fără schimbare
            b = np.clip(a + rng.normal(0, 0.03, a.shape).astype(np.float32), 0, 1)
            l = np.where(l == 255, 255, 0).astype(np.uint8)
        elif r < 0.4:                                 # schimbare sintetică
            k = rng.integers(T)
            y2 = rng.integers(0, max(1, H - t + 1)); x2 = rng.integers(self.c0, max(self.c0 + 1, self.c1 - t + 1))
            donor = self._pad(STACK[k, :, y2:y2+t, x2:x2+t].copy(), b, l)[0]
            m = random_blob(t, t, rng)
            diff = np.abs(donor - b).max(0) > (MIN_CHANGE_DB / 27.0)
            b = np.where(m[None], donor, b)
            l = np.where((l != 255) & m & diff, 1, l).astype(np.uint8)

        # augmentări geometrice comune
        kr = rng.integers(4)
        a, b, l = np.rot90(a, kr, (1, 2)), np.rot90(b, kr, (1, 2)), np.rot90(l, kr)
        if rng.random() < 0.5: a, b, l = a[:, :, ::-1], b[:, :, ::-1], l[:, ::-1]
        if rng.random() < 0.5: a, b = b, a
        # augmentări radiometrice per dată
        a = np.clip(a * rng.uniform(0.95, 1.05) + rng.normal(0, 0.01, a.shape), 0, 1)
        b = np.clip(b * rng.uniform(0.95, 1.05) + rng.normal(0, 0.01, b.shape), 0, 1)
        return (torch.from_numpy(np.ascontiguousarray(a, np.float32)),
                torch.from_numpy(np.ascontiguousarray(b, np.float32)),
                torch.from_numpy(np.ascontiguousarray(l, np.int64)))

    def _pad(self, a, b, l):
        t = self.tile
        ph, pw = t - a.shape[1], t - a.shape[2]
        if ph or pw:
            a = np.pad(a, ((0, 0), (0, ph), (0, pw)))
            b = np.pad(b, ((0, 0), (0, ph), (0, pw)))
            l = np.pad(l, ((0, ph), (0, pw)), constant_values=255)
        if self.split == "val":
            return torch.from_numpy(a), torch.from_numpy(b), torch.from_numpy(l.astype(np.int64))
        return a, b, l

train_ds = ChangeDataset("train", STEPS_EPOCH * BATCH)
val_ds   = ChangeDataset("val", 0)
train_dl = torch.utils.data.DataLoader(train_ds, BATCH, num_workers=8, pin_memory=True,
                                       persistent_workers=True, worker_init_fn=lambda w: np.random.seed(SEED + w))
val_dl   = torch.utils.data.DataLoader(val_ds, BATCH, num_workers=4, pin_memory=True)
print(f"train: {len(train_ds)} mostre/epocă | val: {len(val_ds)} tile-uri")

## 8. Model: Siamese U-Net

Encoder **partajat** pentru ambele date → la fiecare nivel fuzionăm `[f1, f2, |f1−f2|]` → decoder U-Net → 1 canal (logit schimbare).
Intrări: `t1`, `t2` de formă `[B, 2, H, W]` (H, W multiplu de 16).

In [ ]:
class ConvBlock(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.GELU())

class SiameseUNet(nn.Module):
    def __init__(self, in_ch=2, widths=(48, 96, 192, 384, 768)):
        super().__init__()
        self.enc = nn.ModuleList()
        c = in_ch
        for w in widths:
            self.enc.append(ConvBlock(c, w)); c = w
        self.pool = nn.MaxPool2d(2)
        self.fuse = nn.ModuleList([nn.Sequential(nn.Conv2d(3 * w, w, 1, bias=False),
                                                 nn.BatchNorm2d(w), nn.GELU()) for w in widths])
        self.up, self.dec = nn.ModuleList(), nn.ModuleList()
        for wi in range(len(widths) - 1, 0, -1):
            self.up.append(nn.ConvTranspose2d(widths[wi], widths[wi - 1], 2, stride=2))
            self.dec.append(ConvBlock(2 * widths[wi - 1], widths[wi - 1]))
        self.head = nn.Conv2d(widths[0], 1, 1)

    def encode(self, x):
        feats = []
        for k, blk in enumerate(self.enc):
            if k: x = self.pool(x)
            x = blk(x); feats.append(x)
        return feats

    def forward(self, t1, t2):
        f1, f2 = self.encode(t1), self.encode(t2)
        fused = [fz(torch.cat([a, b, (a - b).abs()], 1)) for fz, a, b in zip(self.fuse, f1, f2)]
        x = fused[-1]
        for up, dec, skip in zip(self.up, self.dec, reversed(fused[:-1])):
            x = dec(torch.cat([up(x), skip], 1))
        return self.head(x)

model = SiameseUNet(in_ch=len(POLS)).to(DEVICE).to(memory_format=torch.channels_last)
print(f"Parametri: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")

## 9. Antrenare (bf16 autocast, AdamW, cosine LR, BCE + Dice)

In [ ]:
def loss_fn(logits, y):
    logits = logits[:, 0]
    m = (y != 255).float()
    yt = (y == 1).float()
    bce = (F.binary_cross_entropy_with_logits(logits, yt, reduction="none",
                                              pos_weight=torch.tensor(3.0, device=y.device)) * m).sum() / m.sum().clamp(min=1)
    p = torch.sigmoid(logits) * m
    dice = 1 - (2 * (p * yt).sum() + 1) / (p.sum() + (yt * m).sum() + 1)
    return bce + dice

@torch.no_grad()
def evaluate(model, dl, thr=0.5):
    model.eval(); tp = fp = fn = 0; tot = 0.0; n = 0
    for a, b, y in dl:
        a, b, y = a.to(DEVICE), b.to(DEVICE), y.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            lg = model(a, b)
        tot += loss_fn(lg.float(), y).item(); n += 1
        pr = (torch.sigmoid(lg.float()[:, 0]) > thr); m = y != 255; yt = y == 1
        tp += (pr & yt & m).sum().item(); fp += (pr & ~yt & m).sum().item(); fn += (~pr & yt & m).sum().item()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9); iou = tp / max(tp + fp + fn, 1)
    return dict(loss=tot / max(n, 1), precision=prec, recall=rec, f1=f1, iou=iou)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
train_step_model = torch.compile(model)   # comentează dacă torch.compile dă probleme

history, best_f1 = [], -1
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = 0.0
    for a, b, y in tqdm(train_dl, desc=f"epoca {ep}/{EPOCHS}", leave=False):
        a = a.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        b = b.to(DEVICE, non_blocking=True).to(memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.bfloat16):
            logits = train_step_model(a, b)
        loss = loss_fn(logits.float(), y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); run += loss.item()
    val = evaluate(model, val_dl)
    history.append(dict(epoch=ep, train_loss=run / len(train_dl), **val))
    flag = ""
    if val["f1"] > best_f1:
        best_f1 = val["f1"]; flag = " ⭐"
        torch.save({"model": model.state_dict(), "pols": POLS, "db_range": DB_RANGE, "epoch": ep, "val": val},
                   CKPT_DIR / "best.pt")
    print(f"ep {ep:3d} | train {history[-1]['train_loss']:.4f} | val {val['loss']:.4f} "
          f"| F1 {val['f1']:.3f} IoU {val['iou']:.3f} P {val['precision']:.3f} R {val['recall']:.3f} "
          f"| {time.time()-t0:.0f}s{flag}")

json.dump(history, open(OUT_DIR / "train_history.json", "w"), indent=1)

In [ ]:
h = history
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["train_loss"] for r in h], label="train"); ax[0].plot([r["loss"] for r in h], label="val"); ax[0].set_title("loss"); ax[0].legend()
ax[1].plot([r["f1"] for r in h], label="F1"); ax[1].plot([r["iou"] for r in h], label="IoU"); ax[1].set_title("val metrics"); ax[1].legend()
plt.show()

ck = torch.load(CKPT_DIR / "best.pt", map_location=DEVICE)
model.load_state_dict(ck["model"]); model.eval()
print("Model încărcat din epoca", ck["epoch"], ck["val"])

## 10. Inferență pe toată scena și pe toată seria temporală

Sliding window cu suprapunere + ponderare (fără artefacte la marginea tile-urilor). Pentru fiecare pereche consecutivă `(t_k, t_{k+1})` obținem o hartă de probabilitate, apoi produse agregate:
* **change_count** — de câte ori s-a schimbat fiecare pixel;
* **first_change** — indexul primei date la care s-a schimbat (−1 = niciodată);
* **cumulative** — prima dată vs. ultima dată.

In [ ]:
THRESHOLD = 0.5

@torch.no_grad()
def predict_pair(a, b, tile=512, overlap=128):
    # a, b: np [C,H,W] normalizate -> prob [H,W]
    C, h, w = a.shape
    ph, pw = (-h) % 16, (-w) % 16
    A = torch.from_numpy(np.pad(a, ((0, 0), (0, ph), (0, pw))))[None].to(DEVICE)
    B = torch.from_numpy(np.pad(b, ((0, 0), (0, ph), (0, pw))))[None].to(DEVICE)
    Hp, Wp = A.shape[-2:]
    t = min(tile, Hp, Wp); step = max(16, t - overlap)
    win = torch.outer(torch.hann_window(t, periodic=False), torch.hann_window(t, periodic=False)).to(DEVICE) + 1e-3
    out = torch.zeros(Hp, Wp, device=DEVICE); wsum = torch.zeros_like(out)
    ys = list(range(0, max(Hp - t, 0) + 1, step)); xs = list(range(0, max(Wp - t, 0) + 1, step))
    if ys[-1] != Hp - t: ys.append(Hp - t)
    if xs[-1] != Wp - t: xs.append(Wp - t)
    for y in ys:
        for x in xs:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                p = torch.sigmoid(model(A[..., y:y+t, x:x+t], B[..., y:y+t, x:x+t]).float())[0, 0]
            out[y:y+t, x:x+t] += p * win; wsum[y:y+t, x:x+t] += win
    return (out / wsum)[:h, :w].cpu().numpy()

PROBS = []
for k in tqdm(range(T - 1), desc="inferență perechi consecutive"):
    p = predict_pair(STACK[k], STACK[k + 1])
    p[~(VALID[k] & VALID[k + 1])] = 0
    PROBS.append(p)
PROBS = np.stack(PROBS).astype(np.float32)        # [T-1, H, W]
BIN = PROBS > THRESHOLD

change_count = BIN.sum(0).astype(np.uint8)
first_change = np.where(BIN.any(0), BIN.argmax(0), -1).astype(np.int16)  # index pereche
cumulative = predict_pair(STACK[0], STACK[-1])

pix_km2 = abs(REF_TRANSFORM.a * REF_TRANSFORM.e) / 1e6
for k in range(T - 1):
    print(f"{DATES[k]} → {DATES[k+1]}: {BIN[k].sum()*pix_km2:8.2f} km² schimbați")

In [ ]:
k_max = int(BIN.reshape(T - 1, -1).sum(1).argmax())
fig, ax = plt.subplots(2, 3, figsize=(18, 11))
show(ax[0, 0], STACK[k_max, 0], f"VV {DATES[k_max]}", cmap="gray")
show(ax[0, 1], STACK[k_max + 1, 0], f"VV {DATES[k_max+1]}", cmap="gray")
show(ax[0, 2], PROBS[k_max], "probabilitate schimbare (model)", cmap="inferno", vmin=0, vmax=1)
show(ax[1, 0], cumulative, f"cumulativ {DATES[0]} → {DATES[-1]}", cmap="inferno", vmin=0, vmax=1)
show(ax[1, 1], change_count, "nr. schimbări (frecvență)", cmap="viridis")
fc = np.where(first_change >= 0, first_change, np.nan)
im = ax[1, 2].imshow(fc, cmap="turbo"); ax[1, 2].set_title("prima schimbare (index pereche)"); ax[1, 2].axis("off")
plt.colorbar(im, ax=ax[1, 2], fraction=0.03)
plt.tight_layout(); plt.show()

areas = [BIN[k].sum() * pix_km2 for k in range(T - 1)]
plt.figure(figsize=(12, 3.5)); plt.bar(DATES[1:], areas); plt.xticks(rotation=60)
plt.ylabel("km² schimbați"); plt.title("Evoluția schimbărilor în timp"); plt.tight_layout(); plt.show()

## 11. Export pentru frontend (GeoTIFF + GeoJSON + timeline.json)

* GeoTIFF-uri în proiecția originală (UTM) — pentru analiză/QGIS sau servite ca tile-uri.
* **GeoJSON în EPSG:4326** — poligoane de schimbare per pereche, direct în Leaflet / MapLibre / deck.gl din React.
* `timeline.json` — date, suprafețe schimbate, căi către fișiere — un endpoint Rust îl poate servi direct.

In [ ]:
from rasterio.features import shapes
from shapely.geometry import shape, mapping

PROD_DIR = OUT_DIR / "products"; PROD_DIR.mkdir(exist_ok=True)
profile = dict(driver="GTiff", height=H, width=W, crs=REF_CRS, transform=REF_TRANSFORM,
               compress="deflate", tiled=True, blockxsize=256, blockysize=256)

def write_tif(path, arr, dtype, nodata=None):
    with rasterio.open(path, "w", count=1, dtype=dtype, nodata=nodata, **profile) as dst:
        dst.write(arr.astype(dtype), 1)

def to_geojson(mask, props, min_px=10, simplify_m=15):
    feats = []
    lab, n = ndi.label(mask)
    if n:
        sizes = ndi.sum(mask, lab, range(1, n + 1))
        mask = np.isin(lab, np.where(sizes >= min_px)[0] + 1)
    for geom, val in shapes(mask.astype(np.uint8), mask=mask, transform=REF_TRANSFORM):
        g = shape(geom).simplify(simplify_m)
        if g.is_empty: continue
        g4326 = transform_geom(REF_CRS, "EPSG:4326", mapping(g))
        feats.append({"type": "Feature", "geometry": g4326,
                      "properties": {**props, "area_km2": round(g.area / 1e6, 4)}})
    return {"type": "FeatureCollection", "features": feats}

timeline = {"burst_id": BURST, "aoi_wkt": AOI_WKT, "crs": str(REF_CRS),
            "bounds_wgs84": list(transform_bounds(REF_CRS, "EPSG:4326",
                                                  *rasterio.transform.array_bounds(H, W, REF_TRANSFORM))),
            "dates": DATES, "threshold": THRESHOLD, "pairs": []}

for k in tqdm(range(T - 1), desc="export"):
    name = f"change_{DATES[k]}_{DATES[k+1]}"
    write_tif(PROD_DIR / f"{name}_prob.tif", (PROBS[k] * 255).round(), "uint8")
    gj = to_geojson(BIN[k], {"from": DATES[k], "to": DATES[k + 1]})
    json.dump(gj, open(PROD_DIR / f"{name}.geojson", "w"))
    timeline["pairs"].append({"from": DATES[k], "to": DATES[k + 1],
                              "changed_km2": round(float(BIN[k].sum() * pix_km2), 3),
                              "changed_pct": round(float(BIN[k].mean() * 100), 3),
                              "n_polygons": len(gj["features"]),
                              "prob_tif": f"products/{name}_prob.tif",
                              "geojson": f"products/{name}.geojson"})

write_tif(PROD_DIR / "change_count.tif", change_count, "uint8")
write_tif(PROD_DIR / "first_change.tif", first_change, "int16", nodata=-1)
write_tif(PROD_DIR / "cumulative_prob.tif", (cumulative * 255).round(), "uint8")
json.dump(to_geojson(cumulative > THRESHOLD, {"from": DATES[0], "to": DATES[-1]}),
          open(PROD_DIR / "cumulative.geojson", "w"))
timeline["aggregates"] = {"change_count": "products/change_count.tif",
                          "first_change": "products/first_change.tif",
                          "cumulative_prob": "products/cumulative_prob.tif",
                          "cumulative_geojson": "products/cumulative.geojson"}
json.dump(timeline, open(OUT_DIR / "timeline.json", "w"), indent=1)
print("✅ Produse salvate în", PROD_DIR)

## 12. Export model ONNX pentru backend-ul Rust

* `model.onnx` — intrări `t1`, `t2`: `float32 [N, 2, H, W]` (valori normalizate în [0,1], H și W multiplu de 16); ieșire `change_prob`: `float32 [N, 1, H, W]` (sigmoid deja aplicat).
* `model_meta.json` — tot ce trebuie backend-ului ca să reproducă preprocesarea exact.

In [ ]:
class ExportWrapper(nn.Module):
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, t1, t2): return torch.sigmoid(self.m(t1, t2))

import copy
export_model = ExportWrapper(copy.deepcopy(model)).eval().float().cpu()
dummy = (torch.rand(1, len(POLS), 256, 256), torch.rand(1, len(POLS), 256, 256))
onnx_path = OUT_DIR / "model.onnx"
kw = dict(input_names=["t1", "t2"], output_names=["change_prob"], opset_version=17,
          dynamic_axes={"t1": {0: "N", 2: "H", 3: "W"}, "t2": {0: "N", 2: "H", 3: "W"},
                        "change_prob": {0: "N", 2: "H", 3: "W"}})
try:
    torch.onnx.export(export_model, dummy, str(onnx_path), dynamo=False, **kw)
except TypeError:   # versiuni vechi de torch fără argumentul `dynamo`
    torch.onnx.export(export_model, dummy, str(onnx_path), **kw)

import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(onnx_path)))

# verificare: ONNX vs PyTorch pe o bucată reală, dimensiune diferită de cea de export
hh, ww = (min(H, 384) // 16) * 16, (min(W, 512) // 16) * 16
a = STACK[0, :, :hh, :ww][None]; b = STACK[-1, :, :hh, :ww][None]
sess = ort.InferenceSession(str(onnx_path), providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
o = sess.run(None, {"t1": a, "t2": b})[0]
with torch.no_grad():
    r = export_model(torch.from_numpy(a), torch.from_numpy(b)).numpy()
print("max |ONNX − torch| =", float(np.abs(o - r).max()), "| providers:", sess.get_providers())

meta = {
    "model": "SiameseUNet", "format": "onnx", "opset": 17,
    "inputs": {"t1": "float32[N,C,H,W]", "t2": "float32[N,C,H,W]"},
    "output": {"change_prob": "float32[N,1,H,W], sigmoid applied"},
    "channels": POLS,
    "preprocessing": {
        "source": "OPERA L2 RTC-S1 (gamma0 linear power, 30 m)",
        "speckle_filter": {"type": "lee", "window": 5},
        "to_db": "10*log10(max(x, 1e-5))",
        "normalize": {pol: {"db_min": lo, "db_max": hi, "formula": "clip((db-db_min)/(db_max-db_min),0,1)"}
                      for pol, (lo, hi) in DB_RANGE.items()},
        "nodata": "înlocuit cu mediana scenei înainte de filtru",
    },
    "spatial_multiple": 16, "recommended_tile": 512, "tile_overlap": 128,
    "threshold": THRESHOLD, "pixel_size_m": float(REF_TRANSFORM.a),
    "trained_on": {"burst_id": BURST, "dates": DATES, "aoi_wkt": AOI_WKT},
    "val_metrics": ck["val"], "created": dt.datetime.now(dt.timezone.utc).isoformat(),
}
json.dump(meta, open(OUT_DIR / "model_meta.json", "w"), indent=1)
print("✅", onnx_path, f"({onnx_path.stat().st_size/1e6:.1f} MB) + model_meta.json")

## 13. Arhivă finală (pentru backend)

In [ ]:
import shutil
bundle = shutil.make_archive(str(ROOT / "sar_change_bundle"), "zip", root_dir=OUT_DIR,
                             base_dir=".")
print("📦", bundle)
!ls -la {OUT_DIR}
# from google.colab import files; files.download(bundle)   # download local

## Integrare cu backend-ul Rust (rezumat)

Structura bundle-ului:
```
outputs/
├── model.onnx            ← inferență în Rust (crate `ort`)
├── model_meta.json       ← parametri de preprocesare / prag
├── timeline.json         ← endpoint GET /api/timeline pentru React
├── train_history.json
└── products/
    ├── change_<d1>_<d2>.geojson     ← straturi pe hartă (EPSG:4326)
    ├── change_<d1>_<d2>_prob.tif
    ├── change_count.tif, first_change.tif, cumulative_prob.tif
    └── cumulative.geojson
```

Schiță Rust (`Cargo.toml`: `ort = "2"`, `ndarray = "0.16"`):
```rust
use ort::{session::Session, value::Tensor};
let mut session = Session::builder()?.commit_from_file("outputs/model.onnx")?;
// t1, t2: ndarray::Array4<f32> de formă [1, 2, H, W], normalizate ca în model_meta.json
let outputs = session.run(ort::inputs![
    "t1" => Tensor::from_array(t1)?,
    "t2" => Tensor::from_array(t2)?,
])?;
let prob = outputs["change_prob"].try_extract_array::<f32>()?;  // [1, 1, H, W]
```

Pentru demo cel mai simplu: backend-ul servește direct `timeline.json` + GeoJSON-urile precalculate; inferența ONNX live în Rust e necesară doar pentru zone/date noi.